In [1]:
import pandas as pd
import re

# 1. 기존 노트북(TextData_spoiler_service.ipynb)의 텍스트 정제 함수 정의[cite: 1]
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # URL, 멘션 제거
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    # 한글, 알파벳, 숫자, 공백만 남기기
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
    # 의미 없는 자음/모음 반복 제거
    text = re.sub(r'[ㄱ-ㅎㅏ-ㅣ]+', '', text)
    # 다중 공백 축소
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("1. 3개의 영화 리뷰 파일을 불러오는 중입니다...")
# 파일 로드
df_master = pd.read_csv('movie_reviews_master.csv')
df_kino_main = pd.read_csv('kinolights_final_reviews.csv')
df_kino_12k = pd.read_csv('kinolights_final_reviews_12000.csv')

print(f"   - 마스터 리뷰 개수: {df_master.shape[0]}건")
print(f"   - 키노라이츠 메인 리뷰 개수: {df_kino_main.shape[0]}건")
print(f"   - 키노라이츠 12000 리뷰 개수: {df_kino_12k.shape[0]}건")

# 2. 각 파일의 형식(컬럼명) 일치시키기 및 데이터 추출
# 마스터 파일은 이미 정제된 'cleaned_review' 컬럼을 사용합니다.
df1 = df_master[['movie_title', 'cleaned_review']].copy()
df1 = df1.rename(columns={'cleaned_review': 'text'})

# 키노라이츠 파일들은 원본 'text' 컬럼을 가져와서 정제 함수를 적용합니다.
print("\n2. 키노라이츠 데이터 텍스트 정제 중 (시간이 약간 소요될 수 있습니다)...")
df2 = df_kino_main[['movie_title', 'text']].copy()
df2['text'] = df2['text'].apply(clean_text)

df3 = df_kino_12k[['movie_title', 'text']].copy()
df3['text'] = df3['text'].apply(clean_text)

# 3. 데이터 하나로 통합 (행 방향 합치기)
print("\n3. 모든 데이터를 하나로 통합하는 중...")
merged_df = pd.concat([df1, df2, df3], ignore_index=True)
print(f"   - 단순 통합 직후 총 개수: {merged_df.shape[0]}건")

# 4. 데이터 정제 (결측치 제거 ➔ 중복 제거 ➔ 길이 필터링)
merged_df = merged_df.dropna(subset=['text'])
merged_df = merged_df.drop_duplicates(subset=['text'], keep='first')
print(f"   - 중복 제거 후 개수: {merged_df.shape[0]}건")

# 글자 수가 15자 이상인 유의미한 리뷰만 남기기
merged_df = merged_df[merged_df['text'].str.len() >= 15]
print(f"✨ - 15자 이상 최종 필터링 후 개수: {merged_df.shape[0]}건")

# 5. 최종 마스터 데이터셋 저장하기
output_path = 'final_merged_reviews_all.csv'
merged_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n✅ 1단계 완료! 3개 파일이 합쳐진 통합 마스터 데이터가 '{output_path}'로 저장되었습니다.")

1. 3개의 영화 리뷰 파일을 불러오는 중입니다...
   - 마스터 리뷰 개수: 133482건
   - 키노라이츠 메인 리뷰 개수: 49914건
   - 키노라이츠 12000 리뷰 개수: 697건

2. 키노라이츠 데이터 텍스트 정제 중 (시간이 약간 소요될 수 있습니다)...

3. 모든 데이터를 하나로 통합하는 중...
   - 단순 통합 직후 총 개수: 184093건
   - 중복 제거 후 개수: 180034건
✨ - 15자 이상 최종 필터링 후 개수: 151172건

✅ 1단계 완료! 3개 파일이 합쳐진 통합 마스터 데이터가 'final_merged_reviews_all.csv'로 저장되었습니다.
